# Dual-GPU DB Stage 5 — shufflenet_v2_x1_0 / culture-b / seed 48

Add Output của notebook Stage 1-4 tương ứng làm Kaggle Input. Hai GPU chia sẻ hàng đợi gồm DB fixed, DB Bayesian, DB Bayesian Elite, Random, Top-K và Greedy.

In [ ]:
BACKBONE = 'shufflenet_v2_x1_0'
DATASET_ID = 'culture-b'
DATA_DIR = '/kaggle/working'
SEED = 48
PRIOR_RUN_ID = 'culture-b__shufflenet_v2_x1_0__db__seed_48'
MC_SAMPLES = 32


In [ ]:
from pathlib import Path
source = Path('/kaggle/input/datasets/utkarshsaxenadn/fast-food-classification-dataset/Fast Food Classification V2')
for destination, origin in {'train': 'Train', 'test': 'Test', 'val': 'Valid'}.items():
    path = Path('/kaggle/working') / destination
    if path.is_symlink():
        path.unlink()
    elif path.exists():
        raise FileExistsError(f'Không ghi đè path thật: {path}')
    path.symlink_to(source / origin, target_is_directory=True)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

GIT_COMMIT = 'REPLACE_WITH_PUSHED_COMMIT_SHA'
PROJECT = Path('/kaggle/working/neuro_symbolic_mlops_l2_app')
if not (PROJECT / '.git').exists():
    if PROJECT.exists():
        # Chỉ dọn đúng thư mục clone dở của notebook trước đó.
        shutil.rmtree(PROJECT)
    git_env = os.environ.copy()
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception as error:
        token = None
        print('Không đọc được Kaggle secret GITHUB_TOKEN:', type(error).__name__)
    if token:
        # Truyền auth qua environment để token không xuất hiện trong command/traceback.
        git_env['GIT_CONFIG_COUNT'] = '1'
        git_env['GIT_CONFIG_KEY_0'] = 'http.extraHeader'
        git_env['GIT_CONFIG_VALUE_0'] = f'Authorization: Bearer {token}'
        print('GitHub authentication: Kaggle secret GITHUB_TOKEN')
    else:
        print('GitHub authentication: none (chỉ hoạt động nếu repo public)')
    clone = subprocess.run(
        ['git', 'clone', '--filter=blob:none', 'https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git', str(PROJECT)],
        env=git_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if clone.returncode:
        raise RuntimeError(
            'git clone thất bại. Hãy bật Kaggle Internet và cấu hình '
            'GITHUB_TOKEN nếu repo private. Git stderr: ' + clone.stderr.strip()
        )
subprocess.run(['git', 'fetch', 'origin', GIT_COMMIT, '--depth', '1'], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', GIT_COMMIT], cwd=PROJECT, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip() == GIT_COMMIT


In [ ]:
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!pip install -q torchgfn tensordict dvclive dvc openpyxl
!nvidia-smi -L
import torch
assert torch.cuda.device_count() >= 2, 'Hãy chọn Kaggle Accelerator: GPU T4 x2'


In [ ]:
import logging
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['DISABLE_DVCLIVE'] = '1'
os.environ['DVCLIVE_LOGLEVEL'] = 'ERROR'
os.environ['DVC_NO_ANALYTICS'] = '1'
logging.getLogger('dvclive').setLevel(logging.ERROR)
logging.getLogger('dvc').setLevel(logging.ERROR)
print('Dual-GPU mode: DVCLive disabled; warnings suppressed.')


In [ ]:
import subprocess, sys
from pathlib import Path
output = Path('/kaggle/working/db_stage5_runs') / BACKBONE / DATASET_ID / f'seed_{SEED}'
subprocess.run([
    sys.executable, '-m', 'pipelines.run_dual_gpu_db_stage5_seed',
    '--config', 'params.yaml',
    '--seed', str(SEED),
    '--dataset-id', DATASET_ID,
    '--prior-run-id', PRIOR_RUN_ID,
    '--backbone', BACKBONE,
    '--data-dir', DATA_DIR,
    '--output-dir', str(output),
    '--project-dir', str(PROJECT),
    '--working-dir', '/kaggle/working',
    '--kaggle-input-root', '/kaggle/input',
    '--mc-samples', str(MC_SAMPLES),
], cwd=PROJECT, check=True)
